# GenIDS-NB15: NFStream flow extraction and labeling

This notebook extracts bidirectional flows from the original UNSW-NB15 PCAP files with NFStream, associates the extracted flows with the labels from the original UNSW-NB15 flow table, standardizes the labels used by GenIDS, reduces the benign class, and exports the resulting dataset.

Only the paths and, if necessary, the PCAP filename ranges in the configuration cell should be changed. The source PCAP files and the original UNSW-NB15 flow table are not distributed with this repository.

## 1. Configuration

In [ ]:
from pathlib import Path

PCAP_JANUARY_DIR = Path("/path/to/unsw-nb15/pcap_22_01_15")
PCAP_FEBRUARY_DIR = Path("/path/to/unsw-nb15/pcap_17_02_15")
ORIGINAL_FLOW_FILE = Path("/path/to/unsw-nb15/unsw_original.csv")
INTERMEDIATE_DIR = Path("/path/to/intermediate/nfstream_flows")
OUTPUT_DIR = Path("/path/to/output")
OUTPUT_FILE = OUTPUT_DIR / "GenIDS-NB15.csv"

JANUARY_PCAP_FILES = [f"{index:02}.pcap" for index in range(1, 54)]
FEBRUARY_PCAP_FILES = [f"{index:02}.pcap" for index in range(1, 27)]

IDLE_TIMEOUT = 300
ACTIVE_TIMEOUT = 20
BPF_FILTER = "ip"
TIMEZONE = "Australia/Sydney"
BENIGN_SAMPLE_SIZE = 599_345
RANDOM_STATE = 42

NFSTREAM_DROP_COLUMNS = [
    "content_type",
    "user_agent",
    "server_fingerprint",
    "client_fingerprint",
    "requested_server_name",
]

MATCH_COLUMNS = [
    "src_ip",
    "src_port",
    "dst_ip",
    "dst_port",
    "src2dst_first_seen_ms",
    "src2dst_last_seen_ms",
]

NUMERIC_MATCH_COLUMNS = [
    "src_port",
    "dst_port",
    "src2dst_first_seen_ms",
    "src2dst_last_seen_ms",
]

## 2. Imports and helper functions

In [ ]:
import re

import pandas as pd
import pytz
from nfstream import NFStreamer


def validate_files(directory, filenames, group_name):
    missing = [directory / filename for filename in filenames if not (directory / filename).is_file()]
    if missing:
        formatted = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(f"Missing {group_name} PCAP files:\n{formatted}")


def extract_pcap_group(directory, filenames):
    frames = []
    for position, filename in enumerate(filenames, start=1):
        source = directory / filename
        print(f"[{position}/{len(filenames)}] Extracting {source.name}")
        frame = NFStreamer(
            source=str(source),
            idle_timeout=IDLE_TIMEOUT,
            active_timeout=ACTIVE_TIMEOUT,
            statistical_analysis=True,
            decode_tunnels=True,
            bpf_filter=BPF_FILTER,
        ).to_pandas()
        frames.append(frame)
        print(f"    Extracted flows: {len(frame):,}")
    return pd.concat(frames, ignore_index=True)


def remove_invalid_reference_rows(frame):
    hexadecimal = re.compile(r"^0x[0-9a-fA-F]+$")
    invalid_hex = frame.astype(str).apply(lambda column: column.str.match(hexadecimal)).any(axis=1)
    invalid_port = frame["src_port"].eq("-") | frame["dst_port"].eq("-")
    return frame.loc[~(invalid_hex | invalid_port)].copy()


def standardize_reference_table(frame):
    standardized = frame.rename(columns={
        "srcip": "src_ip",
        "sport": "src_port",
        "dstip": "dst_ip",
        "dsport": "dst_port",
        "Stime": "src2dst_first_seen_ms",
        "Ltime": "src2dst_last_seen_ms",
    }).copy()
    required = set(MATCH_COLUMNS + ["attack_cat", "Label"])
    missing = required.difference(standardized.columns)
    if missing:
        raise ValueError(f"Original flow table is missing columns: {sorted(missing)}")
    standardized = remove_invalid_reference_rows(standardized)
    standardized[NUMERIC_MATCH_COLUMNS] = standardized[NUMERIC_MATCH_COLUMNS].apply(pd.to_numeric, errors="raise").astype("int64")
    standardized["attack_cat"] = standardized["attack_cat"].replace({"0": "Benign"}).fillna("Benign")
    return standardized


def prepare_extracted_flows(frame):
    prepared = frame.drop(columns=NFSTREAM_DROP_COLUMNS, errors="ignore").dropna().copy()
    missing = set(MATCH_COLUMNS).difference(prepared.columns)
    if missing:
        raise ValueError(f"NFStream output is missing columns: {sorted(missing)}")
    prepared[NUMERIC_MATCH_COLUMNS] = prepared[NUMERIC_MATCH_COLUMNS].apply(pd.to_numeric, errors="raise").astype("int64")
    prepared["src2dst_first_seen_ms"] //= 1000
    prepared["src2dst_last_seen_ms"] //= 1000
    timestamps = pd.to_datetime(
        prepared["src2dst_first_seen_ms"], unit="s", utc=True
    ).dt.tz_convert(pytz.timezone(TIMEZONE))
    prepared["date"] = timestamps.dt.strftime("%d/%m/%Y")
    prepared["hours"] = timestamps.dt.strftime("%H:%M")
    return prepared.drop(columns=["id"], errors="ignore").reset_index(drop=True)


def build_label_reference(frame):
    reference = frame[MATCH_COLUMNS + ["attack_cat", "Label"]].copy()
    conflicting = reference.groupby(MATCH_COLUMNS, dropna=False)["attack_cat"].nunique()
    conflicting = conflicting[conflicting > 1]
    if not conflicting.empty:
        raise ValueError(f"Found {len(conflicting)} flow keys associated with conflicting attack labels.")
    return reference.drop_duplicates(subset=MATCH_COLUMNS)


def normalize_attack_category(value):
    normalized = str(value).strip().lower()
    if normalized in {"0", "benign", "normal"}:
        return "benign"
    return normalized


def label_extracted_flows(extracted, reference):
    labels = build_label_reference(reference)
    labeled = extracted.merge(labels, on=MATCH_COLUMNS, how="inner", validate="many_to_one")
    labeled["attack_cat"] = labeled["attack_cat"].map(normalize_attack_category)
    labeled["binary"] = labeled["attack_cat"].map(lambda value: "benign" if value == "benign" else "malign")
    labeled["multiclass"] = labeled["attack_cat"].map(
        lambda value: "benign" if value == "benign" else "ddos" if value == "dos" else "background"
    )
    return labeled.drop(columns=["attack_cat", "Label"]).reset_index(drop=True)


def reduce_benign_class(frame):
    benign = frame.loc[frame["multiclass"] == "benign"]
    non_benign = frame.loc[frame["multiclass"] != "benign"]
    if len(benign) < BENIGN_SAMPLE_SIZE:
        raise ValueError(
            f"BENIGN_SAMPLE_SIZE={BENIGN_SAMPLE_SIZE:,} exceeds the {len(benign):,} matched benign flows."
        )
    sampled = benign.sample(n=BENIGN_SAMPLE_SIZE, random_state=RANDOM_STATE)
    return pd.concat([sampled, non_benign], ignore_index=True)


def class_summary(frame, column):
    return pd.DataFrame({
        "count": frame[column].value_counts(),
        "percentage": frame[column].value_counts(normalize=True).mul(100).round(2),
    })

## 3. Validate inputs

In [ ]:
validate_files(PCAP_JANUARY_DIR, JANUARY_PCAP_FILES, "January")
validate_files(PCAP_FEBRUARY_DIR, FEBRUARY_PCAP_FILES, "February")

if not ORIGINAL_FLOW_FILE.is_file():
    raise FileNotFoundError(f"Original UNSW-NB15 flow table not found: {ORIGINAL_FLOW_FILE}")

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input validation completed successfully.")

## 4. Extract flows from the original PCAP files

In [ ]:
january_flows = extract_pcap_group(PCAP_JANUARY_DIR, JANUARY_PCAP_FILES)
february_flows = extract_pcap_group(PCAP_FEBRUARY_DIR, FEBRUARY_PCAP_FILES)

january_file = INTERMEDIATE_DIR / "nfstream_22_01_15.csv"
february_file = INTERMEDIATE_DIR / "nfstream_17_02_15.csv"
january_flows.to_csv(january_file, index=False)
february_flows.to_csv(february_file, index=False)

print(f"January flows:  {len(january_flows):,}")
print(f"February flows: {len(february_flows):,}")

## 5. Prepare the NFStream output and original label table

In [ ]:
extracted_flows = pd.concat([january_flows, february_flows], ignore_index=True)
prepared_flows = prepare_extracted_flows(extracted_flows)

original_flows = pd.read_csv(ORIGINAL_FLOW_FILE, low_memory=False)
reference_flows = standardize_reference_table(original_flows)

print(f"Extracted flows after cleaning: {len(prepared_flows):,}")
print(f"Reference flows after cleaning: {len(reference_flows):,}")

## 6. Associate NFStream flows with UNSW-NB15 labels

In [ ]:
genids_nb15_full = label_extracted_flows(prepared_flows, reference_flows)
matched_percentage = 100 * len(genids_nb15_full) / len(prepared_flows)

print(f"Matched and labeled flows: {len(genids_nb15_full):,}")
print(f"Matched percentage: {matched_percentage:.2f}%")
display(class_summary(genids_nb15_full, "binary"))
display(class_summary(genids_nb15_full, "multiclass"))

## 7. Reduce the benign class and remove duplicates

In [ ]:
genids_nb15 = reduce_benign_class(genids_nb15_full)
rows_before = len(genids_nb15)
genids_nb15 = genids_nb15.drop_duplicates().reset_index(drop=True)

label_columns = ["binary", "multiclass"]
feature_columns = [column for column in genids_nb15.columns if column not in label_columns]
genids_nb15 = genids_nb15[feature_columns + label_columns]

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal:  {len(genids_nb15):,}")
display(class_summary(genids_nb15, "binary"))
display(class_summary(genids_nb15, "multiclass"))

## 8. Export

In [ ]:
genids_nb15.to_csv(OUTPUT_FILE, index=False)

print(f"Dataset saved to: {OUTPUT_FILE.resolve()}")
print(f"Final shape: {genids_nb15.shape}")
display(genids_nb15.head())